In [1]:
import cv2
import numpy as np

In [2]:
fgbg = cv2.createBackgroundSubtractorMOG2()
capture = cv2.VideoCapture('dataset.mp4')
fps = capture.get(cv2.CAP_PROP_FPS)
print("FPS:", fps)
for i in range(6):
    # Đọc từng frame của video
    ret, frame = capture.read()
    # Nếu không đọc được video (hết video hoặc lỗi) thì thoát vòng lặp
    if not ret:
        break
    # Áp dụng thuật toán trừ nền để lấy foreground mask
    fgMask = fgbg.apply(frame)
cv2.imshow('Frame', frame)
cv2.imshow('FG Mask', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()


print("Sum of intensities in fgMask is : ", np.sum(fgMask))  # ← dùng đúng như đề
print('Frame shape:',frame.shape)


FPS: 24.0
Sum of intensities in fgMask is :  977374
Frame shape: (480, 640, 3)


In [3]:
fgMask = cv2.dilate(fgMask,None,iterations=2)
cv2.imshow('Frame', frame)
cv2.imshow('FG Mask', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()
print("Sum of intensities in fgMask is : ", np.sum(fgMask))

Sum of intensities in fgMask is :  2036269


In [4]:
fgMask = cv2.erode(fgMask, None, iterations=2)
cv2.imshow('Frame', frame)
cv2.imshow('FG Mask', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()
print("Sum of intensities in fgMask is : ", np.sum(fgMask))  # ← dùng đúng như đề


Sum of intensities in fgMask is :  1181423


In [5]:
fgMask = cv2.erode(fgMask, None, iterations=2)
fgMask = cv2.dilate(fgMask, None, iterations=2)
cv2.imshow('Frame', frame)
cv2.imshow('FG Mask', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()
print("Sum of intensities in fgMask is : ", np.sum(fgMask)) 

Sum of intensities in fgMask is :  1076170


In [ ]:
fgbg = cv2.createBackgroundSubtractorMOG2()
capture = cv2.VideoCapture('dataset.mp4')
for i in range(100):
    (grabbed,frame) = capture.read()
    fgMask = fgbg.apply(frame)

_,fgMask = cv2.threshold(fgMask,200,255,cv2.THRESH_BINARY)
fgMask = cv2.dilate(fgMask,None,iterations=2)
fgMask = cv2.erode(fgMask,None,iterations=2)

contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

contours_result = []
for contour in contours:
    M = cv2.moments(contour)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])
        cy = int(M['m01'] / M['m00'])
        contours_result.append([cx, cy])

print('The locations of the foreground objects are:',(contours_result))
cv2.imshow('Frame', frame)
cv2.imshow('FG Mask', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()
print("Sum of intensities in fgMask is : ", np.sum(fgMask)) 

The locations of the foreground objects are: [[5, 233], [354, 210]]
Sum of intensities in fgMask is :  740010


In [ ]:
import cv2

path_to_video = "dataset.mp4"
fgbg = cv2.createBackgroundSubtractorMOG2()
capture = cv2.VideoCapture(path_to_video)

(grabbed, frame) = capture.read()
fgMask = fgbg.apply(frame)
line = 280

contours_previous = []
people_out = 0
people_in = 0
contours_now = []

while True:
    contours_now = []
    (grabbed, frame) = capture.read()

    if not grabbed:
        break

    fgMask = fgbg.apply(frame)

    cv2.putText(frame, str(capture.get(cv2.CAP_PROP_POS_FRAMES)) + "/433", (15, 15),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0))

    fgMask = cv2.threshold(fgMask, 200, 255, cv2.THRESH_BINARY)[1]
    fgMask = cv2.dilate(fgMask, None, iterations=2)
    fgMask = cv2.erode(fgMask, None, iterations=2)

    contours_list, hierarchy = cv2.findContours(fgMask,
                                                cv2.RETR_TREE,
                                                cv2.CHAIN_APPROX_SIMPLE)

    for c in contours_list:
        if cv2.contourArea(c) < 1000:
            continue
        (x, y, w, h) = cv2.boundingRect(c)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
        contours_now.append([x, y])

    if len(contours_previous) == 0:
        contours_previous = contours_now
        continue

    closest_contour_list = []

    # ********* Your code here ***********************************
    for cnt_now in contours_now:        #xét từng người trong frame hiện tại
        min_dist = float('inf')     # khởi tạo khoảng cách nhỏ nhất = vô cực vì chưa tìm được người nào gần nhất
        closest = None
        for cnt_prev in contours_previous:          #duyệt từng người ở frame trước
            if cnt_prev in closest_contour_list:    #nếu người đó ghép với người khác rồi thì bỏ que tránh 2 người cùng ghép cùng 1 người
                continue
            dist = abs(cnt_now[0] - cnt_prev[0]) + abs(cnt_now[1] - cnt_prev[1])        #tính khoảng cách Manhattan
            if dist < min_dist:             
                min_dist = dist
                closest = cnt_prev

        if closest is not None:         #đánh dấu người này đã gặp rồi
            closest_contour_list.append(closest)
            if closest[1] < line and cnt_now[1] >= line:
                people_out += 1
            elif closest[1] >= line and cnt_now[1] < line:
                people_in += 1
    # ************************************************************

    contours_previous = contours_now

    cv2.line(frame, (0, line), (frame.shape[1], line), (0, 255, 255), 2)
    cv2.putText(frame, "People out: " + str(people_out), (15, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    cv2.putText(frame, "People in: " + str(people_in), (14, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow('Frame', frame)
    cv2.imshow('FG Mask', fgMask)

    keyboard = cv2.waitKey(30)
    if keyboard == ord('q') or keyboard == 27:
        break

capture.release()
cv2.destroyAllWindows()
for i in range(1, 5):
    cv2.waitKey(1)

print("People in:", people_in, ", People out:", people_out)


People in: 4 , People out: 12


In [14]:
import cv2
import numpy as np

capture = cv2.VideoCapture('dataset.mp4')

# Đọc 2 frame liên tiếp: frame 5 và frame 6
for i in range(5):
    ret, prev_frame = capture.read()   # frame 1→5 (frame "không có người" hoặc ít người hơn)

ret, curr_frame = capture.read()       # frame 6 (frame "có người")

# Chuyển sang grayscale để dễ tính toán
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)

# Trừ 2 frame → ra vùng khác biệt (người di chuyển)
diff = cv2.absdiff(curr_gray, prev_gray)

# Nhị phân hóa để làm rõ foreground
_, fgMask = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

# Hiển thị kết quả
cv2.imshow('Frame', curr_frame)
cv2.imshow('FG Mask (frame diff)', fgMask)
cv2.waitKey(0)
capture.release()
cv2.destroyAllWindows()
for i in range(1, 5):
    cv2.waitKey(1)

print("Sum of intensities:", np.sum(fgMask))


Sum of intensities: 387090
